In [82]:
import torch
torch.set_printoptions(precision=2, sci_mode=False)

In [83]:
# create a 6 x 6 matrix (random)
mat = torch.randn(6, 6)
mat.shape, mat

(torch.Size([6, 6]),
 tensor([[-1.13, -1.78, -1.12,  0.70, -0.68,  0.84],
         [ 0.34,  0.20, -0.35,  0.88,  0.49, -0.39],
         [-0.14,  0.18,  1.09,  1.24,  0.94,  0.38],
         [ 0.36,  0.10,  1.06,  0.55, -0.11,  0.40],
         [ 0.83,  0.91, -0.37,  0.78,  0.28,  1.00],
         [ 0.16, -1.18,  0.15,  0.04,  0.18, -1.03]]))

In [84]:
# get its softmax to represent what is atten_scores 
mat_norm = torch.softmax(mat / mat.shape[0] ** 0.5, dim=-1)
mat_norm

tensor([[0.12, 0.09, 0.12, 0.25, 0.14, 0.27],
        [0.17, 0.16, 0.13, 0.22, 0.18, 0.13],
        [0.12, 0.14, 0.20, 0.21, 0.19, 0.15],
        [0.16, 0.15, 0.22, 0.18, 0.13, 0.17],
        [0.18, 0.19, 0.11, 0.18, 0.15, 0.20],
        [0.19, 0.11, 0.19, 0.18, 0.20, 0.12]])

In [85]:
# make all the aboce diagonals zeros
casual_mask = torch.zeros_like(mat_norm)
print(casual_mask)
for i in range(mat_norm.shape[0]):
    for j in range(mat_norm.shape[1]):
        if j > i:
            casual_mask[i][j] = -torch.inf
            continue
        casual_mask[i][j] = mat_norm[i][j]

casual_mask

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])


tensor([[0.12, -inf, -inf, -inf, -inf, -inf],
        [0.17, 0.16, -inf, -inf, -inf, -inf],
        [0.12, 0.14, 0.20, -inf, -inf, -inf],
        [0.16, 0.15, 0.22, 0.18, -inf, -inf],
        [0.18, 0.19, 0.11, 0.18, 0.15, -inf],
        [0.19, 0.11, 0.19, 0.18, 0.20, 0.12]])

In [86]:
torch.softmax(torch.tril(mat_norm), dim=-1)

tensor([[0.18, 0.16, 0.16, 0.16, 0.16, 0.16],
        [0.19, 0.19, 0.16, 0.16, 0.16, 0.16],
        [0.17, 0.18, 0.19, 0.15, 0.15, 0.15],
        [0.17, 0.17, 0.18, 0.18, 0.15, 0.15],
        [0.17, 0.18, 0.16, 0.17, 0.17, 0.15],
        [0.17, 0.16, 0.17, 0.17, 0.17, 0.16]])

In [87]:
torch.softmax(casual_mask, dim=-1)

tensor([[1.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.50, 0.50, 0.00, 0.00, 0.00, 0.00],
        [0.32, 0.33, 0.35, 0.00, 0.00, 0.00],
        [0.25, 0.24, 0.26, 0.25, 0.00, 0.00],
        [0.20, 0.21, 0.19, 0.20, 0.20, 0.00],
        [0.17, 0.16, 0.17, 0.17, 0.17, 0.16]])

In [88]:
# Pytorch tricks to do it 
inf_mask = torch.triu(torch.ones(6, 6), diagonal=1)
casual_mask.masked_fill(inf_mask.bool(), -torch.inf)

tensor([[0.12, -inf, -inf, -inf, -inf, -inf],
        [0.17, 0.16, -inf, -inf, -inf, -inf],
        [0.12, 0.14, 0.20, -inf, -inf, -inf],
        [0.16, 0.15, 0.22, 0.18, -inf, -inf],
        [0.18, 0.19, 0.11, 0.18, 0.15, -inf],
        [0.19, 0.11, 0.19, 0.18, 0.20, 0.12]])

In [89]:
torch.softmax(casual_mask, dim=-1)

tensor([[1.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.50, 0.50, 0.00, 0.00, 0.00, 0.00],
        [0.32, 0.33, 0.35, 0.00, 0.00, 0.00],
        [0.25, 0.24, 0.26, 0.25, 0.00, 0.00],
        [0.20, 0.21, 0.19, 0.20, 0.20, 0.00],
        [0.17, 0.16, 0.17, 0.17, 0.17, 0.16]])

In [90]:
# p = 0.1, dropout removes the values (set them 0) & scale the remaining 
# values to the 1 / (1-p)
torch.nn.Dropout(0.1)(torch.softmax(casual_mask, dim=-1))

tensor([[1.11, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.56, 0.55, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.36, 0.39, 0.00, 0.00, 0.00],
        [0.27, 0.00, 0.29, 0.28, 0.00, 0.00],
        [0.23, 0.23, 0.00, 0.23, 0.22, 0.00],
        [0.19, 0.18, 0.19, 0.19, 0.19, 0.18]])

In [91]:
torch.softmax(casual_mask, dim=-1)

tensor([[1.00, 0.00, 0.00, 0.00, 0.00, 0.00],
        [0.50, 0.50, 0.00, 0.00, 0.00, 0.00],
        [0.32, 0.33, 0.35, 0.00, 0.00, 0.00],
        [0.25, 0.24, 0.26, 0.25, 0.00, 0.00],
        [0.20, 0.21, 0.19, 0.20, 0.20, 0.00],
        [0.17, 0.16, 0.17, 0.17, 0.17, 0.16]])

In [92]:
class TestRegister(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.register_buffer("one", torch.tensor([1]))

In [93]:
t_r = TestRegister()
t_r = t_r.to("mps")
t_r.one

tensor([1], device='mps:0')

In [116]:
inputs = torch.randn(2, 6, 3)
inputs

tensor([[[ 0.05, -0.24, -1.00],
         [ 2.31,  0.94,  0.88],
         [-1.76, -0.03,  0.10],
         [ 1.19,  0.41,  0.77],
         [ 1.78,  1.08,  0.24],
         [ 0.25, -1.19,  0.36]],

        [[-0.29, -0.88,  0.75],
         [ 1.49, -0.24, -0.69],
         [-2.15,  0.56, -0.14],
         [ 0.05, -0.81, -0.15],
         [ 0.17,  1.32, -0.86],
         [-1.37, -0.79,  0.11]]])

In [ ]:
class CasAtten(torch.nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_len,
        qkv_bias=False,
        drop_out_p=0.2,
    ):
        super().__init__()
        self.W_Q = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_K = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_V = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.drop_out = torch.nn.Dropout(drop_out_p)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        q = self.W_Q(x)
        k = self.W_K(x)
        v = self.W_V(x)
        atten = q @ k.transpose(1, 2)
        # atten.masked_fill_(self.mask.bool(), -torch.inf)
        # in inference this shape can be changed but in training it stays same mostly
        atten.masked_fill_(self.mask[:x.shape[1], :x.shape[1]].bool(), -torch.inf)
        atten = self.drop_out(torch.softmax(atten / k.shape[-1] ** 0.5, dim=-1))
        return atten @ v

In [129]:
cas_atten = CasAtten(3, 2, 6)
cas_atten(inputs)

tensor([[[ 0.32,  0.28],
         [ 0.17,  0.15],
         [-0.31,  0.07],
         [-0.20, -0.05],
         [-0.40, -0.00],
         [-0.17,  0.05]],

        [[ 0.51, -0.04],
         [ 0.26,  0.21],
         [ 0.10,  0.13],
         [ 0.31,  0.02],
         [-0.00, -0.08],
         [ 0.04, -0.08]]], grad_fn=<UnsafeViewBackward0>)